# Sort circular objects by their orientiation with respect to the central vein
The small vessel channel (anti-CD105 staining with Alexa546) was segmented with Imaris and exported as binary images.
A Fiji macro counted all partical in each slice which have an circular shape (sphericity between 0.9- 1.0)
Here, the exported data will be summarized with respect to the orientation to the central vein

Input: csv-lists with the properties of the circular objects found in the CD105 segmentation

Output: txt-files summarizing the properties of the circular objects such as average area.

In [6]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd

In [7]:
# 3. folder to be processed
path = './CD105/CircularObjects/*_x-z.csv'
save_file_as = 'Circular_xz-plane.txt'

list_filenames = list()
list_properties = list()

In [8]:
# 4. loop over the folder to process all images
for file in glob.glob(path):
    # Get data
    filename = os.path.abspath(file).split(".")[0]
    df = pd.read_csv(file)
    # Column 1: Area , Column 6: Circularity
    # Column 10: Roundness (i.e. 1/aspect ratio), Column 11: solidity (area/convex area)
    
    # Extract required columns
    area = df.iloc[:, 1]  # Python is 0-indexed, so column 2 corresponds to index 1
    
    # Add a new column for apparent diameter
    df['Diameter'] = 2 * np.sqrt(area/np.pi)
    
    # Determine the number of objects (rows)
    nr_objects = len(area)
    
    # Calculate mean, standard deviation, and other statistics for the parameter of interest
    mean_area = df['Area'].mean()
    stdev_area = df['Area'].std()
    min_area = df['Area'].min()
    max_area = df['Area'].max()
    
    mean_diameter = df['Diameter'].mean()
    stdev_diameter = df['Diameter'].std()
    min_diameter = df['Diameter'].min()
    max_diameter = df['Diameter'].max()
    
    mean_circularity = df['Circ.'].mean()
    stdev_circularity = df['Circ.'].std()
    min_circularity = df['Circ.'].min()
    max_circularity = df['Circ.'].max()
    
    props_area = [nr_objects, mean_area, stdev_area, min_area, max_area]
    props_diameter = [mean_diameter, stdev_diameter, min_diameter, max_diameter]
    props_circularity = [mean_circularity, stdev_circularity, min_circularity, max_circularity]
    
    # Append all parameter to a growing list
    list_filenames.append(str(filename))
    list_properties.append(props_area + props_diameter + props_circularity)

    header = 'Filename\t# Objects\tArea mean\tstdev\tmin\tmax\tDiameter mean\tstdev\tmin\tmax\tCircularity mean\tstdev\tmin\tmax'

    # save results as txt file
    np.savetxt(
        save_file_as,
        np.vstack(
            [
                list_filenames,
                np.array(list_properties).T
            ],
        ).T,
        delimiter='\t', fmt="%s", header=header
    )
    